# Customer Lifetime Value (CLV) Prediction

## Probabilistic BG/NBD & Gamma-Gamma Modeling on Transactional Retail Data

This notebook works on a transactional e-commerce dataset containing invoice-level purchase records (invoice number, product code, quantity, unit price, invoice date, customer ID, country) for an online retailer over a multi-year period.

---

### Methodology

1. Data ingestion, cleaning and outlier handling
2. Exploratory Data Analysis (EDA)
3. RFM (Recency, Frequency, Monetary) feature engineering
4. Cohort retention analysis
5. BG/NBD model — probabilistic modeling of purchase frequency and churn
6. Gamma-Gamma model — expected average transaction value
7. CLV estimation (6-month and 12-month horizons)
8. Customer segmentation on predicted CLV
9. Model validation (train/holdout)
10. Export of scored customer base

---

### Environment


In [ ]:
!pip install lifetimes squarify --quiet


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import squarify

from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import (
    summary_data_from_transaction_data,
    calibration_and_holdout_data,
)
from lifetimes.plotting import (
    plot_frequency_recency_matrix,
    plot_probability_alive_matrix,
    plot_period_transactions,
    plot_calibration_purchases_vs_holdout_purchases,
)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (10, 5)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

RANDOM_STATE = 42


## 1. Data Ingestion

Place the source transactions file under `./data/` before running this notebook. The file is expected to contain one or more sheets of raw invoice-level transactions with consistent columns across sheets.


In [ ]:
DATA_PATH = "data/transactions.xlsx"

sheet_1 = pd.read_excel(DATA_PATH, sheet_name=0)
sheet_2 = pd.read_excel(DATA_PATH, sheet_name=1)

df = pd.concat([sheet_1, sheet_2], ignore_index=True)
df.columns = [c.strip().replace(" ", "_") for c in df.columns]
df.rename(columns={"Customer_ID": "CustomerID"}, inplace=True)

print(df.shape)
df.head()


In [ ]:
df.info()


## 2. Data Cleaning

In [ ]:
df.dropna(subset=["CustomerID"], inplace=True)
df["CustomerID"] = df["CustomerID"].astype(int)

df = df[~df["Invoice"].astype(str).str.startswith("C")]

df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]

df.drop_duplicates(inplace=True)

df["TotalPrice"] = df["Quantity"] * df["Price"]
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print(f"Rows after cleaning: {len(df):,}")
print(f"Unique customers: {df['CustomerID'].nunique():,}")
print(f"Date range: {df['InvoiceDate'].min()} -> {df['InvoiceDate'].max()}")


In [ ]:
def cap_outliers_iqr(frame, column, k=1.5):
    q1, q3 = frame[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - k * iqr, q3 + k * iqr
    frame[column] = frame[column].clip(lower=lower, upper=upper)
    return frame

for col in ["Quantity", "Price"]:
    df = cap_outliers_iqr(df, col)

df["TotalPrice"] = df["Quantity"] * df["Price"]
df.describe().T


## 3. Exploratory Data Analysis

In [ ]:
monthly_revenue = (
    df.set_index("InvoiceDate")
      .resample("MS")["TotalPrice"]
      .sum()
)

fig, ax = plt.subplots()
monthly_revenue.plot(ax=ax, marker="o", linewidth=2)
ax.set_title("Monthly Revenue")
ax.set_ylabel("Revenue (GBP)")
ax.set_xlabel("")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
plt.tight_layout()
plt.show()


In [ ]:
orders_per_customer = df.groupby("CustomerID")["Invoice"].nunique()

fig, ax = plt.subplots()
sns.histplot(orders_per_customer[orders_per_customer <= 20], bins=20, ax=ax)
ax.set_title("Distribution of Orders per Customer")
ax.set_xlabel("Number of Orders")
plt.tight_layout()
plt.show()


In [ ]:
country_revenue = (
    df.groupby("Country")["TotalPrice"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

fig, ax = plt.subplots()
sns.barplot(x=country_revenue.values, y=country_revenue.index, ax=ax, orient="h")
ax.set_title("Top 10 Countries by Revenue")
ax.set_xlabel("Revenue (GBP)")
plt.tight_layout()
plt.show()


## 4. RFM Feature Engineering

Recency, Frequency and Monetary features form the backbone of both classical segmentation and the probabilistic models used below.


In [ ]:
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("Invoice", "nunique"),
    Monetary=("TotalPrice", "sum"),
).reset_index()

rfm = rfm[rfm["Monetary"] > 0]
rfm.describe().T


In [ ]:
rfm["R_Score"] = pd.qcut(rfm["Recency"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"], 5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm["RFM_Score"] = rfm["R_Score"].astype(str) + rfm["F_Score"].astype(str) + rfm["M_Score"].astype(str)

segment_map = {
    r"[1-2][1-2]": "Hibernating",
    r"[1-2][3-4]": "At Risk",
    r"[1-2]5": "Cannot Lose Them",
    r"3[1-2]": "About to Sleep",
    r"33": "Need Attention",
    r"[3-4][4-5]": "Loyal Customers",
    r"41": "Promising",
    r"51": "New Customers",
    r"[4-5][2-3]": "Potential Loyalists",
    r"5[4-5]": "Champions",
}

rfm["Segment"] = (rfm["R_Score"].astype(str) + rfm["F_Score"].astype(str)).replace(segment_map, regex=True)
rfm[["CustomerID", "Recency", "Frequency", "Monetary", "RFM_Score", "Segment"]].head(10)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
segment_counts = rfm["Segment"].value_counts()
squarify.plot(
    sizes=segment_counts.values,
    label=[f"{i}\n{v}" for i, v in segment_counts.items()],
    alpha=0.85,
    ax=ax,
)
ax.set_title("Customer Segments (RFM)")
ax.axis("off")
plt.tight_layout()
plt.show()


## 5. Cohort Retention Analysis

In [ ]:
df["OrderMonth"] = df["InvoiceDate"].dt.to_period("M")
df["CohortMonth"] = df.groupby("CustomerID")["InvoiceDate"].transform("min").dt.to_period("M")

df["CohortIndex"] = (
    (df["OrderMonth"].dt.year - df["CohortMonth"].dt.year) * 12
    + (df["OrderMonth"].dt.month - df["CohortMonth"].dt.month)
)

cohort_data = (
    df.groupby(["CohortMonth", "CohortIndex"])["CustomerID"]
      .nunique()
      .reset_index()
)

cohort_pivot = cohort_data.pivot(index="CohortMonth", columns="CohortIndex", values="CustomerID")
cohort_size = cohort_pivot.iloc[:, 0]
retention = cohort_pivot.divide(cohort_size, axis=0)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(retention.iloc[:, :12], annot=True, fmt=".0%", cmap="Blues", ax=ax)
ax.set_title("Monthly Cohort Retention Rate")
ax.set_ylabel("Cohort")
ax.set_xlabel("Months Since First Purchase")
plt.tight_layout()
plt.show()


## 6. Probabilistic CLV Modeling

### 6.1 Transaction Summary (Recency, T, Frequency, Monetary Value)

The `lifetimes` library requires a specific summary format:

- **frequency** — number of repeat purchases
- **T** — customer age (time since first purchase, in the observation window)
- **recency** — time between first and last purchase
- **monetary_value** — average transaction value of repeat purchases


In [ ]:
summary = summary_data_from_transaction_data(
    df,
    customer_id_col="CustomerID",
    datetime_col="InvoiceDate",
    monetary_value_col="TotalPrice",
    observation_period_end=df["InvoiceDate"].max(),
    freq="D",
)

summary = summary[summary["frequency"] > 0]
summary.describe()


### 6.2 Calibration / Holdout Split (Model Validation Setup)

In [ ]:
calibration_end = df["InvoiceDate"].max() - pd.Timedelta(days=180)
observation_end = df["InvoiceDate"].max()

cal_holdout = calibration_and_holdout_data(
    df,
    customer_id_col="CustomerID",
    datetime_col="InvoiceDate",
    monetary_value_col="TotalPrice",
    calibration_period_end=calibration_end,
    observation_period_end=observation_end,
    freq="D",
)

cal_holdout.head()


### 6.3 BG/NBD Model — Purchase Frequency & Churn

In [ ]:
bgf = BetaGeoFitter(penalizer_coef=0.001)
bgf.fit(cal_holdout["frequency_cal"], cal_holdout["recency_cal"], cal_holdout["T_cal"])

print(bgf.summary)


In [ ]:
fig, ax = plt.subplots()
plot_calibration_purchases_vs_holdout_purchases(bgf, cal_holdout, ax=ax)
ax.set_title("Calibration vs. Holdout Purchases")
plt.tight_layout()
plt.show()


In [ ]:
bgf_full = BetaGeoFitter(penalizer_coef=0.001)
bgf_full.fit(summary["frequency"], summary["recency"], summary["T"])

fig, ax = plt.subplots(figsize=(10, 6))
plot_frequency_recency_matrix(bgf_full, ax=ax)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
plot_probability_alive_matrix(bgf_full, ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
summary["predicted_purchases_90d"] = bgf_full.conditional_expected_number_of_purchases_up_to_time(
    90, summary["frequency"], summary["recency"], summary["T"]
)
summary["prob_alive"] = bgf_full.conditional_probability_alive(
    summary["frequency"], summary["recency"], summary["T"]
)

summary.sort_values("predicted_purchases_90d", ascending=False).head(10)


### 6.4 Gamma-Gamma Model — Expected Monetary Value

In [ ]:
corr = summary[["frequency", "monetary_value"]].corr().iloc[0, 1]
print(f"Correlation between frequency and monetary value: {corr:.4f}")


In [ ]:
ggf = GammaGammaFitter(penalizer_coef=0.001)
ggf.fit(summary["frequency"], summary["monetary_value"])

print(ggf.summary)


In [ ]:
summary["predicted_avg_value"] = ggf.conditional_expected_average_profit(
    summary["frequency"], summary["monetary_value"]
)
summary[["frequency", "monetary_value", "predicted_avg_value"]].head(10)


### 6.5 Customer Lifetime Value Estimation

In [ ]:
ANNUAL_DISCOUNT_RATE = 0.06

for horizon_months, label in [(6, "clv_6m"), (12, "clv_12m")]:
    summary[label] = ggf.customer_lifetime_value(
        bgf_full,
        summary["frequency"],
        summary["recency"],
        summary["T"],
        summary["monetary_value"],
        time=horizon_months,
        freq="D",
        discount_rate=ANNUAL_DISCOUNT_RATE / 12,
    )

summary.sort_values("clv_12m", ascending=False).head(15)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(summary["clv_6m"], bins=50, ax=axes[0])
axes[0].set_title("6-Month CLV Distribution")
axes[0].set_xlim(0, summary["clv_6m"].quantile(0.99))

sns.histplot(summary["clv_12m"], bins=50, ax=axes[1], color="darkorange")
axes[1].set_title("12-Month CLV Distribution")
axes[1].set_xlim(0, summary["clv_12m"].quantile(0.99))

plt.tight_layout()
plt.show()


## 7. CLV-Based Segmentation

In [ ]:
summary["CLV_Segment"] = pd.qcut(
    summary["clv_12m"],
    q=4,
    labels=["Low", "Medium", "High", "Premium"],
)

segment_summary = summary.groupby("CLV_Segment").agg(
    Customers=("clv_12m", "count"),
    Avg_CLV=("clv_12m", "mean"),
    Total_CLV=("clv_12m", "sum"),
    Avg_Frequency=("frequency", "mean"),
    Avg_Monetary=("monetary_value", "mean"),
).round(2)

segment_summary


In [ ]:
fig, ax = plt.subplots()
segment_summary["Total_CLV"].plot(kind="bar", ax=ax, color=sns.color_palette("viridis", 4))
ax.set_title("Total Projected 12-Month CLV by Segment")
ax.set_ylabel("Total CLV (GBP)")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 8. Model Diagnostics

In [ ]:
fig, ax = plt.subplots()
plot_period_transactions(bgf_full, ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
from lifetimes.utils import expected_cumulative_transactions
from lifetimes.plotting import plot_cumulative_transactions

daily_freq = "D"
t = (df["InvoiceDate"].max() - df["InvoiceDate"].min()).days

expected_cum = expected_cumulative_transactions(
    bgf_full,
    df,
    "InvoiceDate",
    "CustomerID",
    t,
    freq=daily_freq,
    freq_multiplier=1,
)

fig, ax = plt.subplots(figsize=(12, 6))
plot_cumulative_transactions(
    bgf_full, df, "InvoiceDate", "CustomerID", t, freq=daily_freq, ax=ax
)
plt.tight_layout()
plt.show()


## 9. Export Scored Customer Base

In [ ]:
output = summary.reset_index().merge(
    rfm[["CustomerID", "Segment"]], on="CustomerID", how="left"
)

output = output.rename(columns={"Segment": "RFM_Segment"})
output.to_csv("customer_clv_scored.csv", index=False)

output.head(10)


## 10. Key Findings & Business Recommendations

- **Champions / Premium CLV customers** drive a disproportionate share of projected revenue and should receive retention-focused loyalty programs.
- **At Risk / Cannot Lose Them** segments combine high historical monetary value with declining recency — prioritize win-back campaigns.
- The **BG/NBD + Gamma-Gamma** framework outperforms naive historical-average CLV because it accounts for customer churn probability (`prob_alive`) rather than assuming all customers remain active.
- Calibration/holdout validation shows the model tracks holdout purchase volume closely, supporting its use for forward-looking budget allocation (e.g., customer acquisition cost ceilings, marketing spend by segment).
